### Introduction to Tier 1 Disease Prediction

This notebook focuses on upgrading the AyurFit project from an unsupervised semantic search model (Cosine Similarity) to a structured Two-Tier Machine Learning Architecture. Specifically, it builds the Tier 1 Disease Prediction Model using dense vector embeddings and a Support Vector Machine classifier.

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer
import joblib

c:\Users\Ashu\Desktop\DE\Ayurfit\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Explanation of data loading

An essential first step in the machine learning pipeline is loading the dataset. We will ingest the patient data from our CSV file into a structured Pandas DataFrame to facilitate exploratory analysis and feature engineering.

In [3]:
df = pd.read_csv('../dataset/final ayurfit.csv')
df.head()

,Disease,Hindi Name,Marathi Name,Symptoms,Diagnosis & Tests,Symptom Severity,Duration of Treatment,Medical History,Current Medications,Risk Factors,...,Doshas,Constitution/Prakriti,Diet and Lifestyle Recommendations,Yoga & Physical Therapy,Medical Intervention,Prevention,Prognosis,Complications,Patient Recommendations,Disease_Group
0,Cough,खांसी,खोकला,"sore throat, chest congestion","Chest X-ray, Blood tests, Sputum analysis",Mild to Moderate,1-2 weeks,"Asthma, Respiratory Issues","Cough syrup, Inhalers","Viral infections, Smoking",...,"Vata, Kapha",Vata-Kapha,Avoid cold foods; stay hydrated; consume warm ...,"Anulom Vilom, Pranayama","Cough syrup, Antibiotics",Avoid irritants,Usually mild,"Bronchitis, Pneumonia","Stay hydrated, rest",Respiratory Disorders
1,Diabetes,मधुमेह,मधुमेह,"frequent urination, fatigue","Blood sugar test, HbA1c test",Moderate to High,Lifetime management,Family history of diabetes,"Insulin, Metformin","Obesity, Genetics, Age > 40",...,"Pitta, Kapha",Kapha,Avoid sugary foods; focus on low-GI foods; reg...,"Surya Namaskar, Pranayama","Insulin, Oral meds",Regular exercise,"Chronic, manageable","Retinopathy, Kidney disease","Healthy diet, exercise",Metabolic & Endocrine Disorders
2,Hypertension,उच्च रक्तचाप,उच्च रक्तदाब,"high blood pressure, stress",Blood pressure measurement,High,Lifetime management,"Heart disease, Stroke","Beta-blockers, Diuretics","Family history, Obesity, Age",...,"Pitta, Vata",Pitta,Reduce salt; practice yoga and meditation; avo...,"Surya Namaskar, Meditation",Antihypertensive meds,Salt restriction,Chronic,"Heart failure, Stroke","Limit salt, exercise",Cardiovascular Disorders
3,Migraine,माइग्रेन,डोकेदुखी,"severe headache, nausea","CT scan, MRI, Neurological exam",Moderate to Severe,3-4 days for relief,Family history of migraines,Pain relievers (NSAIDs),"Stress, Hormonal changes",...,"Pitta, Vata",Pitta-Vata,Maintain regular meal times; avoid bright ligh...,"Anulom Vilom, Pranayama","Pain relievers, Botox",Stress management,Variable,"Stroke, Anxiety, Depression","Sleep, avoid triggers",Neurological Disorders
4,Arthritis,गठिया,आर्थरायटीस,"joint pain, swelling","X-ray, MRI, Blood tests (RA factor)",Moderate to Severe,Variable,"Joint pain, Obesity",Pain relievers (NSAIDs),"Age, Joint overuse, Genetics",...,"Vata, Kapha",Vata,Consume anti-inflammatory foods; stay active; ...,"Yoga for flexibility, Strength","NSAIDs, Steroids",Weight management,Chronic,"Joint deformity, Mobility issues","Exercise, joint care",Musculoskeletal Disorders


### Task 1: Fix the Data Imbalance (Data Cleaning Cell)

Before label encoding, we count the frequency of each Disease. We filter the DataFrame to only keep rows where the Disease appears at least 2 times. This prevents the `UndefinedMetricWarning` during the train/test split and ensures we have enough data for a balanced split.

In [4]:
df = df.dropna(subset=['Symptoms', 'Disease_Group'])
disease_counts = df['Disease_Group'].value_counts()
df = df[df['Disease_Group'].isin(disease_counts[disease_counts >= 2].index)]

### Explanation of Target Encoding

Machine learning classifiers require numerical targets. We utilize `LabelEncoder` to convert the categorical `Disease` labels into a numerical format, establishing a clear mapping that can be reversed during inference.

In [5]:
label_encoder = LabelEncoder()
df['Disease_Encoded'] = label_encoder.fit_transform(df['Disease_Group'])

### Explanation of utilizing GPU for NLP Feature Extraction

We transition from traditional bag-of-words approaches to state-of-the-art transformer models. We initialize the `all-MiniLM-L6-v2` SentenceTransformer, explicitly utilizing the GPU to significantly accelerate the generation of semantic embeddings.

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

Using device: cpu


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5418.73it/s]


### Explanation of generating dense vector embeddings

We transform the raw textual `Symptoms` into a rich mathematical representation. The transformer encodes these texts into 384-dimensional dense vectors, capturing semantic meaning and nuance, which will serve as our primary feature matrix (X).

In [7]:
X = model.encode(df['Symptoms'].tolist(), show_progress_bar=True)
y = df['Disease_Encoded'].values

Batches: 100%|██████████| 40/40 [00:04<00:00,  8.74it/s]


### Task 2: Update Train/Test Split

We update the `train_test_split` function to include `stratify=y`. This ensures every disease is proportionally represented in both the training and testing sets. We use `test_size=0.3` to ensure the mathematical conditions for stratifying hundreds of classes are met.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### Support Vector Machine Training

Support Vector Machines (SVM), specifically `LinearSVC`, are highly effective in high-dimensional spaces, making them an excellent choice for 384-dimensional text embeddings. The model learns optimal hyperplanes to separate the distinct disease classes based on their symptom embeddings.

In [9]:
svm_classifier = LinearSVC(random_state=42, dual=False)
svm_classifier.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rando

### Task 3: Print Overall Accuracy

We generate the `classification_report` on the present classes to eliminate 0-support noise. Afterwards, we specifically calculate and print the overall `accuracy_score` as a clean percentage.

In [10]:
y_pred = svm_classifier.predict(X_test)
present_classes = np.unique(np.concatenate((y_test, y_pred)))
target_names_present = label_encoder.inverse_transform(present_classes)
print(classification_report(y_test, y_pred, labels=present_classes, target_names=target_names_present, zero_division=0))

overall_accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {overall_accuracy * 100:.1f}%")

                                                            precision    recall  f1-score   support

                                      Acanthosis Nigricans       1.00      1.00      1.00         1
                                                      Acne       0.00      0.00      0.00         1
                                                Acromegaly       1.00      1.00      1.00         1
                                       Acute Kidney Injury       1.00      1.00      1.00         1
                                         Addison's Disease       0.50      1.00      0.67         1
                                     Adrenal Insufficiency       0.00      0.00      0.00         1
                                              Alkaptonuria       1.00      1.00      1.00         2
                                                 Allergies       1.00      1.00      1.00         1
                                                  Alopecia       1.00      1.00      1.00         1

### Task 4: Create a Manual Testing Function

We define a function called `predict_custom_symptoms(symptoms_text)` to test out the model dynamically on new strings. It takes string input, uses the transformer to encode it into a 384-dimensional array, passes it to the SVM for prediction, and applies inverse-transform to extract the human-readable disease name. We also calculate the prediction confidence by applying a softmax over the model's decision function logits.

In [11]:
def predict_custom_symptoms(symptoms_text):
    # 1. Generate the 384-dimensional embedding for the input string
    embedding = model.encode([symptoms_text], show_progress_bar=False)
    
    # 2. Predict the encoded label using the SVM model
    predicted_encoded = svm_classifier.predict(embedding)
    
    # 3. Calculate a confidence percentage using the decision function (Softmax)
    decision_scores = svm_classifier.decision_function(embedding)
    exp_scores = np.exp(decision_scores - np.max(decision_scores)) # Prevent overflow
    probabilities = exp_scores / exp_scores.sum(axis=1, keepdims=True)
    confidence = np.max(probabilities) * 100
    
    # 4. Inverse-transform the label back into the readable Disease string
    predicted_disease = label_encoder.inverse_transform(predicted_encoded)[0]
    
    # 5. Print the input symptoms, predicted disease, and confidence
    print(f"Input Symptoms: '{symptoms_text}'")
    print(f"Predicted Disease: {predicted_disease} (Confidence: {confidence:.1f}%)\n")
    
    return predicted_disease, confidence

### Task 5: Run the Tests

We call `predict_custom_symptoms()` with 3 distinct manual test cases based on typical dataset symptoms to verify that our NLP + SVM pipeline correctly classifies general inputs.

In [12]:
predict_custom_symptoms('severe chest pain and shortness of breath')
predict_custom_symptoms('frequent urination and extreme thirst')
predict_custom_symptoms('itchy red skin rash')

Input Symptoms: 'severe chest pain and shortness of breath'
Predicted Disease: Coronary Artery Disease (Confidence: 0.8%)

Input Symptoms: 'frequent urination and extreme thirst'
Predicted Disease: Diabetes Mellitus (Confidence: 0.9%)

Input Symptoms: 'itchy red skin rash'
Predicted Disease: Skin Allergy (Confidence: 0.6%)



('Skin Allergy', np.float64(0.6025508931713761))

In [13]:
predict_custom_symptoms('Heavy blood flow and feeling pain in uterus')

Input Symptoms: 'Heavy blood flow and feeling pain in uterus'
Predicted Disease: Menstrual Disorders (Confidence: 0.8%)



('Menstrual Disorders', np.float64(0.8079762717931992))

In [14]:
predict_custom_symptoms('Cough and Cold with fever')

Input Symptoms: 'Cough and Cold with fever'
Predicted Disease: Asparagine Synthetase Deficiency (Confidence: 0.6%)



('Asparagine Synthetase Deficiency', np.float64(0.6393619510730406))

In [15]:
predict_custom_symptoms('Back pain and Body Pain')

Input Symptoms: 'Back pain and Body Pain'
Predicted Disease: Scoliosis (Confidence: 0.6%)



('Scoliosis', np.float64(0.555201565056273))

### Task 6: Export Models

Finally, we serialize our updated SVM classifier and label encoder using `joblib`. Exporting these artifacts enables seamless integration into the production backend.

In [16]:
joblib.dump(svm_classifier, './sys-dis imp files/tier1_svm_model.joblib')
joblib.dump(label_encoder, './sys-dis imp files/disease_label_encoder.joblib')

['./sys-dis imp files/disease_label_encoder.joblib']